# 19 — Baseline lấy mẫu đơn giản trên LightGBM

Bốn nhóm: không trọng số, class weight cân bằng, random undersampling, random oversampling. Cùng dense features/cache, cấu hình LightGBM và early stopping của nhóm cây cũ. Mỗi nhóm thử 3 num_leaves tại seed 42, rồi thêm 2 seed: **20 fit duy nhất**.

Tỷ lệ minority/majority sau lấy mẫu cố định **0,1**; ROS giữ mọi dòng gốc và thêm bản sao fraud, RUS giữ mọi fraud. Không bù nghịch đảo xác suất. Validation/test giữ nguyên. Resampling chỉ là baseline tham chiếu; hướng cải tiến vẫn là weighting.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "supplement" / "reporting.py").is_file():
    raise RuntimeError("Mở notebook từ BAF_project hoặc BAF_project/notebooks sau khi giải nén.")

sys.path.insert(0, str(ROOT / "supplement"))
import reporting

reporting.setup(ROOT)
print("Python:", sys.executable)
print("Project:", ROOT)
print("Supplement:", reporting.VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Supplement: advisor-20261030-v1


### Chọn campaign đã chạy notebook 11–16

In [2]:
# Giữ None nếu chỉ có một campaign.
# Có nhiều campaign: điền Path hoặc chuỗi đường dẫn campaign cụ thể.
CAMPAIGN_DIR = None
CONFIG = {"campaign": str(CAMPAIGN_DIR) if CAMPAIGN_DIR else None, "ratio": 0.1}
print(CONFIG)

{'campaign': None, 'ratio': 0.1}


### Thực thi và tự lưu notebook có output

In [3]:
worker_setup = """
import os
import sys
from pathlib import Path

ROOT = Path(os.environ["BAF_PROJECT_ROOT"])
RUN_DIR = Path(os.environ["BAF_RUN_DIR"])
sys.path.insert(0, str(ROOT / "supplement"))

import reporting
import training

reporting.setup(ROOT)
CONFIG = reporting.read_json(RUN_DIR / "manifest.json")["config"]
print("CONFIG", CONFIG)
"""

worker_run = """
table = training.run_baf(
    ROOT,
    RUN_DIR,
    'sampling',
    campaign=CONFIG["campaign"],
    ratio=CONFIG["ratio"],
)
print(table.to_string(index=False))
print("ADVISOR_SAMPLING_READY", RUN_DIR)
"""

RUN_DIR = reporting.launch(
    ROOT,
    'sampling',
    CONFIG,
    [worker_setup, worker_run],
)
print("Kết quả:", RUN_DIR)
print("Notebook có output:")
for path in sorted((RUN_DIR / "attempts").glob("*/executed.ipynb")):
    print(path)


RUN_DIR D:\UIT\DS317\BAF_project\results\advisor_runs\sampling_1827c85af012
BEGIN_STAGE 1
CONFIG {'campaign': None, 'ratio': 0.1}
END_STAGE 1
BEGIN_STAGE 2
FIT_BEGIN lgbm_none_c00_s42 {'method': 'lgbm_none', 'sampling': 'none', 'ratio': 0.1, 'candidate': 0, 'leaves': 15, 'family': 'lsr_lgbm', 'feature': 'dense'}
d:\UIT\.venvs\ml-gpu\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
FIT_COMPLETE lgbm_none_c00_s42 validation_AP 0.17717053663012244
FIT_BEGIN lgbm_none_c01_s42 {'method': 'lgbm_none', 'sampling': 'none', 'ratio': 0.1, 'candidate': 1, 'leaves': 31, 'family': 'lsr_lgbm', 'feature': 'dense'}
d:\UIT\.venvs\ml-gpu\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set,